## 1. Setup & Load Libraries

In [1]:
import pandas as pd
import numpy as np
import ast
import faiss
import pickle
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

e:\DS300-UIT-RecommenderSystem\DS300-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Dataset

In [2]:
# Load dataset
DATA_PATH = r"E:\DS300-UIT-RecommenderSystem\Finalproject\data\all_recipes_final.csv"
df = pd.read_csv(DATA_PATH)

print(f"Loaded dataset: {len(df)} recipes")
print(f"Columns: {df.columns.tolist()}")

Loaded dataset: 10335 recipes
Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']


## 3. Hybrid Chunking Strategy

- Metadata = 1 sentence
- Nguyên liệu theo NHÓM (5 items/group) = 2-3 sentences
- Bước nấu theo PHASE (3 steps/group) = 3-4 sentences
- Mô tả = 1 sentence

In [12]:
import ast
import pandas as pd


def parse_list_field(field_value):
    """
    Parse string / list field to Python list safely.
    """
    if pd.isna(field_value):
        return []

    if isinstance(field_value, list):
        return field_value

    if isinstance(field_value, str):
        try:
            parsed = ast.literal_eval(field_value)
            if isinstance(parsed, list):
                return parsed
            return []
        except (ValueError, SyntaxError):
            # fallback: split by comma
            return [s.strip() for s in field_value.split(",") if s.strip()]

    return []


def hybrid_chunking(row):
    """
    Hybrid Chunking Strategy (Production-ready)

    Structure:
    - Metadata: 1 chunk
    - Ingredients: group ~5 items, avoid small tail chunks
    - Steps: group ~3 steps, avoid isolated steps
    - Description: 1 chunk
    """
    sentences = []

    # ------------------------------------------------------------------
    # 1. Metadata
    # ------------------------------------------------------------------
    meta_parts = []

    if pd.notna(row.get("type_of_food")):
        meta_parts.append(str(row["type_of_food"]).strip())

    if pd.notna(row.get("title")):
        meta_parts.append(str(row["title"]).strip())

    if pd.notna(row.get("cook_time")):
        meta_parts.append(f"thời gian {row['cook_time']}")

    if pd.notna(row.get("num_of_people")):
        meta_parts.append(f"Số người ăn: {row['num_of_people']}")

    if meta_parts:
        sentences.append(". ".join(meta_parts))

    # ------------------------------------------------------------------
    # 2. Ingredients (group by ~5, min 3)
    # ------------------------------------------------------------------
    ingredients = parse_list_field(row.get("ingredients"))

    if ingredients:
        chunk_size = 5
        min_chunk = 3

        i = 0
        while i < len(ingredients):
            # merge small tail into previous chunk
            if len(ingredients) - i < min_chunk and sentences:
                sentences[-1] += ", " + ", ".join(ingredients[i:])
                break

            chunk = ingredients[i:i + chunk_size]
            sentences.append("Nguyên liệu: " + ", ".join(chunk))
            i += chunk_size

    # ------------------------------------------------------------------
    # 3. Steps (group by ~3, min 2)
    # ------------------------------------------------------------------
    steps = parse_list_field(row.get("step"))

    if steps:
        chunk_size = 3
        min_chunk = 2

        i = 0
        while i < len(steps):
            # merge isolated tail steps
            if len(steps) - i < min_chunk and sentences:
                sentences[-1] += " → " + " → ".join(steps[i:])
                break

            chunk = steps[i:i + chunk_size]
            sentences.append(" → ".join(chunk))
            i += chunk_size

    # ------------------------------------------------------------------
    # 4. Description
    # ------------------------------------------------------------------
    if pd.notna(row.get("description")):
        desc = str(row["description"]).strip()
        if desc:
            sentences.append(desc)

    # 5. Notes / Tips (list[str] → single semantic chunk)
    notes = parse_list_field(row.get("note"))

    if notes:
        note_text = " | ".join(notes)
        sentences.append("Lưu ý: " + note_text)

    return sentences


In [13]:
test_df = df[:100]

for index, row in test_df.iterrows():
    chunks = hybrid_chunking(row)
    print(f"Recipe: {index}")
    for i, chunk in enumerate(chunks):
        print(f" Chunk {i+1}: {chunk}")
    print("\n")

Recipe: 0
 Chunk 1: Món Tết. Cách muối dưa hành truyền thống. thời gian 45 phút. Số người ăn: 8-10 người
 Chunk 2: Nguyên liệu: 1 kg hành củ tươi, Tro bếp hoặc nước vo gọa, Muối hạt, đường, Cà rốt trang trí (tùy chọn), Lọ sạch
 Chunk 3: Bước 1: Chọn hành củ: Nên chọn hành củ ta bánh tẻ, vừa phải, cầm chắc tay, tròn căng mọng, màu sắc tươi đều (tím nhạt hoặc trắng). Tránh mua hành ấn vào mềm, chảy nước hoặc mốc là đã hỏng. Chỉ nên lựa củ vừa phải, không nên to quá. → Bước 2: Ngâm khử mùi hăng của hành: Theo kinh nghiệm dân gian và trong sách ''Thế vị tân biên'' xuất bản năm 1925 đề cập để khử hăng, giúp hành giòn và trắng nên ngâm với nước tro bếp 2 - 3 ngày. Có nhà dùng nước vo gạo ngâm cũng có tác dụng tương tự. → Bước 3: Nhặt rễ, ngâm nước muối: Dùng dao nhỏ sắc, cắt gần sát rễ (không cắt hết), bóc lớp vỏ già ngoài. Sau đó, ngâm hành vào nước muối loãng ngâm khoảng 30 phút. Việc này giúp khử hành bớt hăng và khử khuẩn để khi ngâm hành không bị nổi váng, úng nhớt. Nếu muốn tăng thêm m

## 4. Generate Sentences for All Recipes

In [14]:
# Generate sentences for all recipes
all_dish_sentences = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing recipes"):
    sentences = hybrid_chunking(row)
    all_dish_sentences.append(sentences)

# Statistics
total_sentences = sum(len(s) for s in all_dish_sentences)
avg_sentences = total_sentences / len(all_dish_sentences)
sentence_counts = [len(s) for s in all_dish_sentences]

print(f"\nCHUNKING RESULTS:")
print(f"   Total recipes:        {len(all_dish_sentences):,}")
print(f"   Total sentences:      {total_sentences:,}")
print(f"   Average/recipe:       {avg_sentences:.1f} sentences")
print(f"   Min/recipe:           {min(sentence_counts)}")
print(f"   Max/recipe:           {max(sentence_counts)}")
print(f"   Median/recipe:        {np.median(sentence_counts):.1f}")

Processing recipes: 100%|██████████| 10335/10335 [00:02<00:00, 4912.65it/s]


CHUNKING RESULTS:
   Total recipes:        10,335
   Total sentences:      68,784
   Average/recipe:       6.7 sentences
   Min/recipe:           2
   Max/recipe:           22
   Median/recipe:        7.0


In [15]:
all_dish_sentences[:5]  # Show first 5 for brevity

[['Món Tết. Cách muối dưa hành truyền thống. thời gian 45 phút. Số người ăn: 8-10 người',
  'Nguyên liệu: 1 kg hành củ tươi, Tro bếp hoặc nước vo gọa, Muối hạt, đường, Cà rốt trang trí (tùy chọn), Lọ sạch',
  "Bước 1: Chọn hành củ: Nên chọn hành củ ta bánh tẻ, vừa phải, cầm chắc tay, tròn căng mọng, màu sắc tươi đều (tím nhạt hoặc trắng). Tránh mua hành ấn vào mềm, chảy nước hoặc mốc là đã hỏng. Chỉ nên lựa củ vừa phải, không nên to quá. → Bước 2: Ngâm khử mùi hăng của hành: Theo kinh nghiệm dân gian và trong sách ''Thế vị tân biên'' xuất bản năm 1925 đề cập để khử hăng, giúp hành giòn và trắng nên ngâm với nước tro bếp 2 - 3 ngày. Có nhà dùng nước vo gạo ngâm cũng có tác dụng tương tự. → Bước 3: Nhặt rễ, ngâm nước muối: Dùng dao nhỏ sắc, cắt gần sát rễ (không cắt hết), bóc lớp vỏ già ngoài. Sau đó, ngâm hành vào nước muối loãng ngâm khoảng 30 phút. Việc này giúp khử hành bớt hăng và khử khuẩn để khi ngâm hành không bị nổi váng, úng nhớt. Nếu muốn tăng thêm màu sắc bắt mắt, tỉa thêm ch

In [16]:
# Test with one sample
print(f"Example (Recipe 0): {df.iloc[0]['title']}")
print(f"Sentences: {len(all_dish_sentences[0])}")
for i, sent in enumerate(all_dish_sentences[0], 1):
    print(f"[{i}] {sent}")

Example (Recipe 0): Cách muối dưa hành truyền thống
Sentences: 5
[1] Món Tết. Cách muối dưa hành truyền thống. thời gian 45 phút. Số người ăn: 8-10 người
[2] Nguyên liệu: 1 kg hành củ tươi, Tro bếp hoặc nước vo gọa, Muối hạt, đường, Cà rốt trang trí (tùy chọn), Lọ sạch
[3] Bước 1: Chọn hành củ: Nên chọn hành củ ta bánh tẻ, vừa phải, cầm chắc tay, tròn căng mọng, màu sắc tươi đều (tím nhạt hoặc trắng). Tránh mua hành ấn vào mềm, chảy nước hoặc mốc là đã hỏng. Chỉ nên lựa củ vừa phải, không nên to quá. → Bước 2: Ngâm khử mùi hăng của hành: Theo kinh nghiệm dân gian và trong sách ''Thế vị tân biên'' xuất bản năm 1925 đề cập để khử hăng, giúp hành giòn và trắng nên ngâm với nước tro bếp 2 - 3 ngày. Có nhà dùng nước vo gạo ngâm cũng có tác dụng tương tự. → Bước 3: Nhặt rễ, ngâm nước muối: Dùng dao nhỏ sắc, cắt gần sát rễ (không cắt hết), bóc lớp vỏ già ngoài. Sau đó, ngâm hành vào nước muối loãng ngâm khoảng 30 phút. Việc này giúp khử hành bớt hăng và khử khuẩn để khi ngâm hành không bị nổi

## 5. Load Vietnamese SBERT Model

Sử dụng `keepitreal/vietnamese-sbert`

In [17]:
# Load Vietnamese SBERT model
model = SentenceTransformer('keepitreal/vietnamese-sbert')
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

   Embedding dimension: 768


## 6. Encode All Sentences → Multi-vector Embeddings

In [18]:
def encode_multi_vector_dishes(dish_sentences_list, model, batch_size=64):
    """
    Encode all sentences into embeddings
    
    Returns:
        dish_embeddings_list: List of numpy arrays, mỗi món = list of vectors
    """
    # Flatten all sentences
    all_sentences = []
    sentence_counts = []
    
    for sentences in dish_sentences_list:
        all_sentences.extend(sentences)
        sentence_counts.append(len(sentences))
    
    print(f"Encoding {len(all_sentences):,} sentences...")
    
    # Encode all at once
    all_embeddings = model.encode(
        all_sentences,
        show_progress_bar=True,
        batch_size=batch_size
    )
    
    # Split back to per-dish
    dish_embeddings_list = []
    start_idx = 0
    for count in sentence_counts:
        end_idx = start_idx + count
        dish_embeddings = all_embeddings[start_idx:end_idx]
        dish_embeddings_list.append(dish_embeddings)
        start_idx = end_idx
    
    print(f"Encoding completed!")
    print(f"   {len(dish_embeddings_list):,} recipes encoded")
    print(f"   {sentence_counts[0]}-{max(sentence_counts)} vectors per recipe")
    
    return dish_embeddings_list, all_embeddings, sentence_counts

In [20]:
# Encode test dish
test_dish_sentences = all_dish_sentences[:3]
dish_embeddings, flat_embeddings, sentence_counts = encode_multi_vector_dishes(
    test_dish_sentences, 
    model,
    batch_size=64
)

print(f"EMBEDDING RESULTS:")
print(f"   Total recipes:        {len(dish_embeddings):,}")
print(f"   Total vectors:        {len(flat_embeddings):,}")
print(f"   Vector dimension:     {flat_embeddings.shape[1]}")
print(f"   Example recipe 0:     {len(dish_embeddings[0])} vectors")

Encoding 18 sentences...


Batches: 100%|██████████| 1/1 [00:04<00:00,  4.47s/it]

Encoding completed!
   3 recipes encoded
   5-7 vectors per recipe
EMBEDDING RESULTS:
   Total recipes:        3
   Total vectors:        18
   Vector dimension:     768
   Example recipe 0:     5 vectors


In [25]:
sentence_counts

[5, 6, 7]

In [ ]:
# Encode all dishes
dish_embeddings, flat_embeddings, sentence_counts = encode_multi_vector_dishes(
    all_dish_sentences, 
    model,
    batch_size=64
)

print(f"EMBEDDING RESULTS:")
print(f"   Total recipes:        {len(dish_embeddings):,}")
print(f"   Total vectors:        {len(flat_embeddings):,}")
print(f"   Vector dimension:     {flat_embeddings.shape[1]}")
print(f"   Example recipe 0:     {len(dish_embeddings[0])} vectors")

## 7. Build FAISS Index

Sử dụng **IndexFlatIP** (Inner Product) vì embeddings đã được normalize

In [ ]:
# Normalize embeddings for cosine similarity
print("Normalizing embeddings...")
faiss.normalize_L2(flat_embeddings)

# Build FAISS index
print("Building FAISS index...")
dimension = flat_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product = Cosine similarity when normalized
index.add(flat_embeddings)

print(f"✅ FAISS index built successfully!")
print(f"   Index size: {index.ntotal:,} vectors")
print(f"   Dimension:  {dimension}")

## 8. Create Mapping: Vector Index → Recipe Info

In [ ]:
# Create mapping: flat_index → (recipe_idx, sentence_idx)
vector_to_recipe_mapping = []
for recipe_idx, count in enumerate(sentence_counts):
    for sentence_idx in range(count):
        vector_to_recipe_mapping.append({
            'recipe_idx': recipe_idx,
            'sentence_idx': sentence_idx,
            'sentence_text': all_dish_sentences[recipe_idx][sentence_idx]
        })

print(f"✅ Mapping created: {len(vector_to_recipe_mapping):,} entries")

In [ ]:
#Example
print(f"Example mapping entry 0:")
print(f"   Recipe index: {vector_to_recipe_mapping[0]['recipe_idx']}")
print(f"   Sentence index: {vector_to_recipe_mapping[0]['sentence_idx']}")
print(f"   Text: {vector_to_recipe_mapping[0]['sentence_text'][:100]}...")

## 9. Save to Disk

In [ ]:
# Save FAISS index
faiss.write_index(index, "../data/food_recipes_hybrid.index")
print("✅ Saved FAISS index: food_recipes_hybrid.index")

# Save mapping
with open("../data/vector_to_recipe_mapping.pkl", "wb") as f:
    pickle.dump(vector_to_recipe_mapping, f)
print("✅ Saved mapping: vector_to_recipe_mapping.pkl")

# Save recipe metadata
df.to_csv("../data/recipes_metadata.csv", index=False, encoding='utf-8-sig')
print("✅ Saved metadata: recipes_metadata.csv")

# Save config
config = {
    'total_recipes': len(df),
    'total_vectors': len(flat_embeddings),
    'avg_sentences_per_recipe': avg_sentences,
    'dimension': dimension,
    'model_name': 'keepitreal/vietnamese-sbert',
    'chunking_strategy': 'hybrid'
}
with open("../data/embedding_config.pkl", "wb") as f:
    pickle.dump(config, f)
print("✅ Saved config: embedding_config.pkl")

## 10. Search Function - Query → Top 20 Results

In [ ]:
def search_recipes(query, model, index, vector_mapping, df, top_k=20, verbose=True):
    """
    Search recipes using semantic similarity
    
    Args:
        query: User's search query (Vietnamese)
        model: SentenceTransformer model
        index: FAISS index
        vector_mapping: Vector to recipe mapping
        df: Recipe metadata dataframe
        top_k: Number of results to return
        verbose: Print detailed results
    
    Returns:
        DataFrame with top_k recipes and similarity scores
    """
    # 1. Encode query
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    
    # 2. Search in FAISS (top 100 vectors để aggregate)
    distances, indices = index.search(query_embedding, min(100, index.ntotal))
    
    # 3. Aggregate scores by recipe (MaxSim strategy)
    recipe_scores = {}
    recipe_matched_sentences = {}
    
    for dist, idx in zip(distances[0], indices[0]):
        recipe_idx = vector_mapping[idx]['recipe_idx']
        sentence_text = vector_mapping[idx]['sentence_text']
        
        if recipe_idx not in recipe_scores:
            recipe_scores[recipe_idx] = dist
            recipe_matched_sentences[recipe_idx] = [(dist, sentence_text)]
        else:
            # Keep max similarity (MaxSim)
            recipe_scores[recipe_idx] = max(recipe_scores[recipe_idx], dist)
            recipe_matched_sentences[recipe_idx].append((dist, sentence_text))
    
    # 4. Sort by score and get top_k
    sorted_recipes = sorted(recipe_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    # 5. Create results dataframe
    results = []
    for recipe_idx, score in sorted_recipes:
        recipe = df.iloc[recipe_idx]
        
        # Get top matched sentence
        matched_sents = sorted(recipe_matched_sentences[recipe_idx], reverse=True)
        top_sentence = matched_sents[0][1] if matched_sents else ""
        
        results.append({
            'recipe_idx': recipe_idx,
            'similarity': float(score),
            'title': recipe['title'],
            'type_of_food': recipe['type_of_food'],
            'cook_time': recipe['cook_time'],
            'description': recipe['description'],
            'matched_sentence': top_sentence
        })
    
    results_df = pd.DataFrame(results)
    
    # 6. Print results
    if verbose:
        print(f"Query: '{query}'")
        print(f"Top {len(results_df)} Results:")
        print("="*100)
        for idx, row in results_df.iterrows():
            print(f"\n{idx+1}. [{row['similarity']:.4f}] {row['title']}")
            print(f"   Loại: {row['type_of_food']} | Thời gian: {row['cook_time']}")
            if pd.notna(row['description']):
                print(f"   Mô tả: {row['description'][:150]}...")
            print(f"   Matched: {row['matched_sentence'][:120]}...")
    
    return results_df

## 11. Test Search Function

In [ ]:
# Test queries
test_queries = [
    "Món ăn có thịt bò nấu nhanh",
    "Món Tết truyền thống của miền Bắc",
    "Canh chua cá cho bữa trưa",
    "Món chay thanh đạm dễ làm"
]

# Run tests
for query in test_queries:
    print("\n" + "="*100)
    results = search_recipes(
        query=query,
        model=model,
        index=index,
        vector_mapping=vector_to_recipe_mapping,
        df=df,
        top_k=5,
        verbose=True
    )
    print("="*100)

## 12. Interactive Search

In [ ]:
# Interactive search
print("INTERACTIVE SEARCH MODE")
print("Enter your query (type 'quit' to exit):\n")

while True:
    query = input("\n>>> Query: ").strip()
    
    if query.lower() in ['quit', 'exit', 'q']:
        print("👋 Goodbye!")
        break
    
    if not query:
        continue
    
    results = search_recipes(
        query=query,
        model=model,
        index=index,
        vector_mapping=vector_to_recipe_mapping,
        df=df,
        top_k=20,
        verbose=True
    )